# Book Recommendation Engine using KNN

Filter infrequent readers and books, then recommend five neighbors using sparse cosine distance. Results are listed from the fifth-nearest to the nearest, matching the challenge example.

Prepared with AI assistance for an agentic AI class assignment. Starter notebook and original challenge tests: [freeCodeCamp](https://github.com/freeCodeCamp/boilerplate-book-recommendation-engine). The supplied final test cell is preserved. Run all cells from top to bottom.

In [1]:
import os
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '2')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '2')
DATA_DIR = Path(os.environ.get('FCC_DATA_DIR', 'data')).resolve()
ARTIFACT_DIR = Path(os.environ.get('FCC_ARTIFACT_DIR', 'artifacts')).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def data_file(name, relative_url, extract=False):
    # The official freeCodeCamp CDN repository is a public mirror of the datasets.
    path = DATA_DIR / name
    if not path.exists() or not path.stat().st_size:
        url = 'https://raw.githubusercontent.com/freeCodeCamp/cdn/main/build/project-data/' + relative_url
        with urlopen(url, timeout=120) as response:
            contents = response.read()
        temporary = path.with_suffix(path.suffix + '.part')
        temporary.write_bytes(contents)
        temporary.replace(path)
    if extract:
        with ZipFile(path) as archive:
            for member in archive.namelist():
                (DATA_DIR / member).resolve().relative_to(DATA_DIR)
            archive.extractall(DATA_DIR)
    return path

In [2]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [3]:
data_file('book-crossings.zip', 'books/book-crossings.zip', extract=True)
books_filename = DATA_DIR / 'BX-Books.csv'
ratings_filename = DATA_DIR / 'BX-Book-Ratings.csv'

In [4]:
df_books = pd.read_csv(books_filename, encoding='ISO-8859-1', sep=';',
    header=0, names=['isbn','title','author'], usecols=[0,1,2],
    dtype={'isbn':'str','title':'str','author':'str'})
df_ratings = pd.read_csv(ratings_filename, encoding='ISO-8859-1', sep=';',
    header=0, names=['user','isbn','rating'],
    dtype={'user':'int32','isbn':'str','rating':'float32'})

In [5]:
user_counts = df_ratings['user'].value_counts()
book_counts = df_ratings['isbn'].value_counts()
selected = df_ratings[
    df_ratings['user'].isin(user_counts[user_counts >= 200].index)
    & df_ratings['isbn'].isin(book_counts[book_counts >= 100].index)
]
selected = selected.merge(df_books[['isbn','title']].drop_duplicates('isbn'), on='isbn')
book_ratings = selected.pivot_table(index='title', columns='user', values='rating',
                                  aggfunc='mean').fillna(0).astype('float32')
ratings_matrix = csr_matrix(book_ratings.to_numpy())
model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=6)
model.fit(ratings_matrix)
title_to_row = {title: index for index, title in enumerate(book_ratings.index)}
print(f'Fitted cosine KNN: {ratings_matrix.shape[0]} titles, {ratings_matrix.shape[1]} readers')

Fitted cosine KNN: 673 titles, 888 readers


In [6]:
def get_recommends(book=''):
    if book not in title_to_row:
        raise ValueError('Choose a title in the filtered Book-Crossings dataset.')
    query = title_to_row[book]
    distances, indices = model.kneighbors(ratings_matrix[query], n_neighbors=6)
    nearest = [(int(i), float(d)) for i, d in zip(indices[0], distances[0]) if i != query][:5]
    recommendations = [[str(book_ratings.index[i]), d] for i, d in reversed(nearest)]
    return [book, recommendations]

In [7]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2): 
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", 0.8016210794448853], ['The Weight of Water', 0.7708583474159241], ['The Surgeon', 0.7699410915374756], ['I Know This Much Is True', 0.7677075266838074], ['The Lovely Bones: A Novel', 0.7234864234924316]]]
You passed the challenge! 🎉🎉🎉🎉🎉


In [8]:
recommendations = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
assert len(recommendations[1]) == 5
assert len({title for title, distance in recommendations[1]}) == 5
assert all(title != recommendations[0] and 0 <= distance <= 2 for title, distance in recommendations[1])
print(get_recommends('The Queen of the Damned (Vampire Chronicles (Paperback))'))

['The Queen of the Damned (Vampire Chronicles (Paperback))', [['Catch 22', 0.793983519077301], ['The Witching Hour (Lives of the Mayfair Witches)', 0.7448656558990479], ['Interview with the Vampire', 0.7345068454742432], ['The Tale of the Body Thief (Vampire Chronicles (Paperback))', 0.5376338362693787], ['The Vampire Lestat (Vampire Chronicles, Book II)', 0.5178412199020386]]]
